# v0.20 Causal ICT/M1 Research Lab

This notebook reproduces the research-only BTC/ETH experiment from official Binance Vision archives. It never sends orders and must not be interpreted as evidence of a profitable strategy.

In [ ]:
import os, pathlib, subprocess, sys
REPO = pathlib.Path('/content/modular-crypto-trading-bot')
BRANCH = 'research-v20-ict-m1'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, 'https://github.com/parsa314/modular-crypto-trading-bot.git', str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[dev]'], check=True)

In [ ]:
# Download and verify official monthly Binance spot archives.
import hashlib
from pathlib import Path
from urllib.error import HTTPError
from urllib.request import Request, urlopen
import pandas as pd

cache = Path('/content/data/cache')
cache.mkdir(parents=True, exist_ok=True)
base = 'https://data.binance.vision/data/spot/monthly/klines'
for symbol in ('BTCUSDT', 'ETHUSDT'):
    for month in pd.period_range('2020-01', '2025-12', freq='M'):
        name = f'{symbol}-4h-{month}.zip'
        target = cache / name
        if target.exists():
            continue
        url = f'{base}/{symbol}/4h/{name}'
        try:
            payload = urlopen(Request(url, headers={'User-Agent': 'v20-ict-m1-research'}), timeout=60).read()
            checksum = urlopen(Request(url + '.CHECKSUM', headers={'User-Agent': 'v20-ict-m1-research'}), timeout=30).read().decode().split()[0]
        except HTTPError as exc:
            if exc.code == 404:
                continue
            raise
        assert hashlib.sha256(payload).hexdigest().lower() == checksum.lower()
        target.write_bytes(payload)
print('archive files:', len(list(cache.glob('*.zip'))))

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_ict_m1_v20.py'], check=True)
env = dict(os.environ, PYTHONPATH=str(REPO))
for symbol in ('BTCUSDT', 'ETHUSDT'):
    subprocess.run([
        sys.executable, 'scripts/run_ict_m1_lab_v20.py',
        '--archive-cache', str(cache), '--symbol', symbol, '--timeframe', '4h',
        '--bars', '20000', '--ichimoku-gate', 'trend',
        '--output-dir', f'/content/artifacts/v20-{symbol.lower()}'
    ], env=env, check=True)

In [ ]:
from IPython.display import display
for symbol in ('btcusdt', 'ethusdt'):
    print(symbol.upper(), 'full period')
    display(pd.read_csv(f'/content/artifacts/v20-{symbol}/summary.csv'))
    print(symbol.upper(), 'period split')
    display(pd.read_csv(f'/content/artifacts/v20-{symbol}/period_summary.csv'))

## Interpretation lock

Positive cells with small trade counts are **insufficient evidence**, not signals. Promotion requires registered multi-timeframe replication, uncertainty intervals, multiple-testing control, and forward paper evidence. `live_execution` remains `false`.